In [62]:
import math
import argparse
import numpy as np
from pathlib import Path

import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from protgen.data import IGNORE_INDEX, ProteinDataset
from protgen.models.transformer import ProteinTransformer

In [63]:
class ProteinClassifier(nn.Module):
    """
    The classifier with frozen transformer + a two heads: generation (use for guided sequence generation) and classification.
    """

    def __init__(self, transf_frozen, lm_head, class_head):
        super().__init__() #initialize the nn.Module class (parent)

        self.transf_frozen = transf_frozen #ProteinTransformer with output layer = nn.Identity
        self.lm_head = lm_head # the predicted next amino acid (dim 23)
        self.class_head = class_head # the predicted class (dim 3)

        def forward(self, x):
            h = self.transf_frozen(x) # running the frozen transformer
            return self.class_head(h) # and then run class_head on that to get protein class scores per position
        
        def lm_logits(self, x):
            return self.lm_head(self.tranf_frozen(x)) # next token scores - for guidance

        # make sure the frozen parts' dropout is turned off
        def train(self, mode=True):
            super().train(mode) # switch to the nn.Module train mode
            self.transf_frozen.eval() #but keep the other parts frozen (eval mode)
            self.lm_head.eval() #but keep the other parts frozen (eval mode)
            return self



In [64]:
def initialize_classifier(
      map_location="cpu",
      model_path = str,
      finetune_entire_model=False,
      number_classes = int, 
      weights_only=True # restricts loading to plain tensors
      ):
    
    print(
    f"Loading model '{model_path}', with "
    f"finetune_entire_model={finetune_entire_model}, changing the "
    f"last layer to output {number_output_classes} logits."
    )

    model = ProteinTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=0.1,
    max_seq_len=max_seq_len,
    )

    checkpoint_file = Path("/Users/kristinetoftjohansen/Desktop/DeepLearning/02456-autoregressive-project/trained_models/trained_transformer_models/stage1/d128_h4_l3_lr0.001_drop0.1_e60_s0.pth")
    model.load_state_dict(torch.load(checkpoint_file, map_location=map_location))
    
    num_model_parameters = 0

    # counts number of parameters
    for p in model.parameters():
        if p.requires_grad:
            num_model_parameters += p.numel()

    # if finetine_entire_model is set to False, the parameters will be frozen
    if not finetune_entire_model:
        for name, param in model.named_parameters():
            param.requires_grad = False

    # save the original output layer, which is a linear layer that outputs the logits for the next amino acid (128 to 23 aas)
    lm_head = model.output_layer

    # make model(x) return the hidden states
    model.output_layer = nn.Identity()

    # adding the classification head
    class_head_features = 64 # also try 32 and 128
    class_head = nn.Sequential(
        nn.Linear(in_features=d_model, out_features=class_head_features), 
        nn.ReLU(), #activation function
        nn.Dropout(0.1), #applying dropout again for more robust learning
        nn.Linear(in_features=class_head_features, out_features=number_output_classes) #128 input features to this layer (d_model)
    )

    # # add the last layer as a classification to one out of three classes
    # model.output_layer = nn.Linear(in_features=d_model, out_features=number_output_classes) #128 input features to this layer (d_model)

    # # Make the last layer trainable.
    # model.output_layer.weight.requires_grad_()
    # model.output_layer.bias.requires_grad_()

    # num_trainable_model_parameters = 0
    # for p in model.parameters():
    #     if p.requires_grad:
    #         num_trainable_model_parameters += p.numel()

    # classifier = ProteinClassifier(model, lm_head, class_head)

    return model, lm_head, class_head, {
        "num_model_parameters": num_model_parameters,
    }

In [66]:

# ####################################
# ### 0. LOAD TRAINED TRANSFORMER ####
# ####################################

vocab_size = 23 # 20 amino acids and <PAD>, <BOS> and <EOS>
d_model = 128 # For every position there's a 128-dimensional vector to carry information, more dimensions means more capacity to represent each position. 
num_heads = 4 # splits d_model dimensions into chunks (d_k = d_model / n_heads)
d_ff = 512
num_layers = 3 # depth
max_len = 512 
max_seq_len = 129 # the maxlength of the dataset sequences (here 127 amino acids (or <PAD>) + <BOS> and <EOS>)
dropout = 0.1
number_output_classes = 3 # number of protein classes

# define the model from the pretrained transformer
model, lm_head, class_head, data = initialize_classifier(map_location = "cpu", number_classes=number_output_classes, finetune_entire_model=False)
classifier = ProteinClassifier(model, lm_head, class_head)

# only hand the trainable parameters to the optimizer
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, classifier.parameters()), lr=1e-3)

print(model)
print("Number of model parameters:", data["num_model_parameters"])
# print("Number of trainable parameters:", data["num_trainable_model_parameters"])

# this part should be as in the transformer
# device = torch.device('cuda')  # use cuda or cpu
# model = model.to(device)




Loading model '<class 'str'>', with finetune_entire_model=False, changing the last layer to output 3 logits.
ProteinTransformer(
  (embedding): Embedding(23, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (layers): ModuleList(
    (0-2): 3 x DecoderBlock(
      (self_attn): MultiHeadedAttention(
        (linears): ModuleList(
          (0-3): 4 x Linear(in_features=128, out_features=128, bias=True)
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (feed_forward): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplac

## Getting an overview

In [70]:
# printing the classifier
print(classifier)

# printing all tensors to see which can learn
for name, p in classifier.named_parameters():
    print(f"{name:50s} {str(tuple(p.shape)):15s} trainable={p.requires_grad}")

ProteinClassifier(
  (transf_frozen): ProteinTransformer(
    (embedding): Embedding(23, 128, padding_idx=0)
    (positional_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-2): 3 x DecoderBlock(
        (self_attn): MultiHeadedAttention(
          (linears): ModuleList(
            (0-3): 4 x Linear(in_features=128, out_features=128, bias=True)
          )
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=512, out_features=128, bias=True)
        )
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    